## Imports

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    upper,
    trim,
    to_date,
    current_timestamp,
    row_number,
    lit,
    when,
    regexp_replace,
    lpad
)
from pyspark.sql.window import Window

## Bronze table

In [0]:
bronze_df = spark.table("bronze_addresses")
silver_customers = spark.table("silver_customers")

display(bronze_df.limit(10))
print(f"Bronze rows: {bronze_df.count():,}")

address_id,customer_id,address_line_1,suburb,state,postcode,country,is_primary,created_at,updated_at,source_system
ADDR0000001,CRM0000001,179/90 Ray Row,Longton,SA,2142,Australia,true,2024-08-26,2024-11-12,crm
ADDR0000002,CRM0000002,287 Kelsey Roadside,Barbarafurt,TAS,2516,Australia,true,2023-12-25,2026-07-29,crm
ADDR0000003,CRM0000003,024 Oneal Quays,North Clintonmouth,NT,2309,Australia,true,2023-02-26,2025-09-11,crm
ADDR0000004,CRM0000004,Suite 215 945 Emily Broadway,New Troystad,TAS,2955,Australia,true,2024-02-19,2027-05-14,crm
ADDR0000005,CRM0000005,91/21 Audrey Gate,Port Scottton,ACT,2600,Australia,true,2026-05-19,2028-10-23,crm
ADDR0000006,CRM0000006,205 Jennifer Port,Sharonmouth,ACT,2630,Australia,true,2022-04-23,2023-01-03,crm
ADDR0000007,CRM0000007,0 Jennifer Street,Leeshire,QLD,2956,Australia,true,2022-07-05,2025-04-15,crm
ADDR0000008,CRM0000008,650/823 Elizabeth Garden,Lake Jacob,SA,2994,Australia,true,2025-03-10,2028-01-11,crm
ADDR0000009,CRM0000009,Unit 07 0 Anthony Rest,Lake Josephberg,NSW,2676,Australia,true,2023-07-26,2023-11-03,crm
ADDR0000010,CRM0000010,013 Ashlee Meander,Port Ronald,SA,2616,Australia,true,2023-09-21,2024-08-25,crm


Bronze rows: 101,000


## Silver transformation

In [0]:
VALID_STATES = ["NSW", "VIC", "QLD", "WA", "SA", "TAS", "ACT", "NT"]

standardised_df = (
        bronze_df
        .withColumn("address_line_1", lower(trim(col("address_line_1"))))
        .withColumn("suburb", lower(trim(col("suburb"))))
        .withColumn("state", upper(trim(col("state"))))
        .withColumn("postcode",lpad(trim(col("postcode").cast("string")), 4, "0"))
        .withColumn("country", lower(trim(col("country"))))
        .withColumn("is_primary", lower(trim(col("is_primary"))))
        .withColumn("created_at", to_date(col("created_at")))
        .withColumn("updated_at", to_date(col("updated_at")))
        .withColumn("_silver_processed_at", current_timestamp())
    )

window_spec = Window.partitionBy("address_id").orderBy(col("updated_at").desc_nulls_last())

deduped_df = (
        standardised_df
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

customer_lookup = (
        silver_customers
        .select("customer_id")
        .dropDuplicates(["customer_id"])
        .withColumn("customer_exists", lit(True))
    )

addresses_with_customer_check = (
        deduped_df
        .join(
            customer_lookup,
            on="customer_id",
            how="left"
        )
    )

quality_df = (
    addresses_with_customer_check
    .withColumn(
        "_has_future_updated_at",
        col("updated_at") > current_timestamp()
    )
    .withColumn(
        "_is_valid",
        when(col("address_id").isNull(), lit(False))
        .when(col("customer_id").isNull(), lit(False))
        .when(col("customer_exists").isNull(), lit(False))
        .when(~col("state").isin(VALID_STATES), lit(False))
        .when(col("postcode").isNull(), lit(False))
        .when(~col("postcode").rlike(r"^[0-9]{4}$"), lit(False))
        .otherwise(lit(True))
    )
    .withColumn(
        "_rejection_reason",
        when(col("address_id").isNull(), lit("address_id is null"))
        .when(col("customer_id").isNull(), lit("customer_id is null"))
        .when(col("customer_exists").isNull(), lit("customer_id does not exist in silver customers"))
        .when(~col("state").isin(VALID_STATES), lit("invalid state"))
        .when(col("postcode").isNull(), lit("postcode is null"))
        .when(~col("postcode").rlike(r"^[0-9]{4}$"), lit("invalid postcode"))
        .otherwise(lit(None))
    )
    )

valid_addresses = quality_df.filter(col("_is_valid") == True)
rejected_addresses = quality_df.filter(col("_is_valid") == False)

## Delta tables

In [0]:
(
    valid_addresses
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_addresses")
)

(
    rejected_addresses
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("rejected_addresses")
)

print("Silver Addresses completed")
print(f"Bronze rows: {bronze_df.count():,}")
print(f"After deduplication: {deduped_df.count():,}")
print(f"Valid addresses: {valid_addresses.count():,}")
print(f"Rejected addresses: {rejected_addresses.count():,}")

Silver Addresses completed
Bronze rows: 101,000
After deduplication: 100,000
Valid addresses: 97,546
Rejected addresses: 2,454


In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|  bronze_addresses|      false|
| default|  bronze_customers|      false|
| default|rejected_addresses|      false|
| default|rejected_customers|      false|
| default|  silver_addresses|      false|
| default|  silver_customers|      false|
+--------+------------------+-----------+

